# Link Prediction in Actor Co-occurrence Networks
## Kaggle Competition - Actor Network Analysis

This notebook addresses the task of predicting missing edges in an actor co-occurrence network using both graph-theoretical features and node textual information.

## 1. Libraries and Setup

In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
SEED = 42

## 2. Data Loading

In [ ]:
# Load training set
with open("train.txt", "r") as f:
    reader = csv.reader(f)
    train_set = list(reader)
train_set = [element[0].split(" ") for element in train_set]
train_df = pd.DataFrame(train_set, columns=['source', 'target', 'label'])
train_df['source'] = train_df['source'].astype(int)
train_df['target'] = train_df['target'].astype(int)
train_df['label'] = train_df['label'].astype(int)

print(f"Training set shape: {train_df.shape}")
print(f"Label distribution:")
print(train_df['label'].value_counts())
print(f"\nFirst few rows:")
print(train_df.head())

In [ ]:
# Load test set
with open("test.txt", "r") as f:
    reader = csv.reader(f)
    test_set = list(reader)
test_set = [element[0].split(" ") for element in test_set]
test_df = pd.DataFrame(test_set, columns=['source', 'target'])
test_df['source'] = test_df['source'].astype(int)
test_df['target'] = test_df['target'].astype(int)

print(f"Test set shape: {test_df.shape}")
print(f"First few rows:")
print(test_df.head())

In [ ]:
# Load node features
node_features = pd.read_csv("node_information.csv", header=None)
print(f"Node features shape: {node_features.shape}")
print(f"Number of features per node: {node_features.shape[1] - 1}")
print(f"Number of nodes: {node_features.shape[0]}")
print(f"\nFirst few rows and columns:")
print(node_features.iloc[:5, :10])

## 3. Build Network Graph from Training Data

In [ ]:
# Create a graph from positive edges in training set
positive_edges = train_df[train_df['label'] == 1][['source', 'target']].values

G = nx.Graph()
# Add all nodes from node_information.csv
G.add_nodes_from(range(node_features.shape[0]))
# Add edges from positive training examples
G.add_edges_from(positive_edges)

print(f"Graph statistics:")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Graph density: {nx.density(G):.4f}")
print(f"Number of connected components: {nx.number_connected_components(G)}")

## 4. Feature Engineering
### 4.1 Graph-Theoretical Features

In [ ]:
def compute_graph_features(G, source, target):
    """
    Compute graph-based features for a node pair (source, target).
    
    Features based on common network science approaches:
    - Jaccard similarity: measures overlap in neighborhoods
    - Adamic-Adar index: weights common neighbors by their degree
    - Preferential attachment: likelihood of connection based on degrees
    - Common neighbors count: raw count of shared neighbors
    - Shortest path length: topological distance
    """
    
    features = {}
    
    # 1. Number of common neighbors
    if source in G and target in G:
        common_neighbors = len(list(nx.common_neighbors(G, source, target)))
    else:
        common_neighbors = 0
    features['common_neighbors'] = common_neighbors
    
    # 2. Jaccard Similarity
    if source in G and target in G:
        neighbors_source = set(G.neighbors(source))
        neighbors_target = set(G.neighbors(target))
        union = len(neighbors_source | neighbors_target)
        features['jaccard'] = len(neighbors_source & neighbors_target) / union if union > 0 else 0
    else:
        features['jaccard'] = 0
    
    # 3. Adamic-Adar Index
    if source in G and target in G:
        adamic_adar = sum([1.0/np.log(G.degree(z)) for z in nx.common_neighbors(G, source, target) if G.degree(z) > 1])
    else:
        adamic_adar = 0
    features['adamic_adar'] = adamic_adar
    
    # 4. Preferential Attachment: degree(source) * degree(target)
    source_degree = G.degree(source) if source in G else 0
    target_degree = G.degree(target) if target in G else 0
    features['pref_attachment'] = source_degree * target_degree
    
    # 5. Degree product (normalized)
    features['degree_source'] = source_degree
    features['degree_target'] = target_degree
    
    # 6. Shortest path length
    if source in G and target in G:
        try:
            shortest_path = nx.shortest_path_length(G, source, target)
        except nx.NetworkXNoPath:
            shortest_path = np.inf
    else:
        shortest_path = np.inf
    features['shortest_path'] = shortest_path if shortest_path != np.inf else 999
    
    return features

print("Graph feature functions defined successfully.")

In [ ]:
# Compute graph features for training set
print("Computing graph features for training set...")
graph_features_train = []
for idx, row in train_df.iterrows():
    features = compute_graph_features(G, row['source'], row['target'])
    graph_features_train.append(features)
    if (idx + 1) % 2000 == 0:
        print(f"  Processed {idx + 1} pairs")

graph_features_train_df = pd.DataFrame(graph_features_train)
print(f"\nGraph features shape: {graph_features_train_df.shape}")
print(f"\nFeature statistics:")
print(graph_features_train_df.describe())

In [ ]:
# Compute graph features for test set
print("Computing graph features for test set...")
graph_features_test = []
for idx, row in test_df.iterrows():
    features = compute_graph_features(G, row['source'], row['target'])
    graph_features_test.append(features)
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1} pairs")

graph_features_test_df = pd.DataFrame(graph_features_test)
print(f"\nGraph features test shape: {graph_features_test_df.shape}")

### 4.2 Node Textual Features

In [ ]:
# Prepare node feature matrix (columns 1 onwards are the actual features)
node_feature_matrix = node_features.iloc[:, 1:].values
print(f"Node feature matrix shape: {node_feature_matrix.shape}")
print(f"Feature matrix dtype: {node_feature_matrix.dtype}")
print(f"\nFeature statistics:")
print(f"Min value: {node_feature_matrix.min()}")
print(f"Max value: {node_feature_matrix.max()}")
print(f"Mean value: {node_feature_matrix.mean():.4f}")

In [ ]:
def compute_text_features(source_features, target_features):
    """
    Compute features based on node textual information.
    
    Features:
    - Cosine similarity between feature vectors
    - L2 distance between vectors
    - Dot product (element-wise similarity measure)
    - Feature overlap (number of shared non-zero features)
    """
    features = {}
    
    # 1. Cosine similarity
    source_norm = np.linalg.norm(source_features)
    target_norm = np.linalg.norm(target_features)
    if source_norm > 0 and target_norm > 0:
        cosine_sim = np.dot(source_features, target_features) / (source_norm * target_norm)
    else:
        cosine_sim = 0
    features['cosine_similarity'] = cosine_sim
    
    # 2. L2 distance (Euclidean)
    l2_distance = np.linalg.norm(source_features - target_features)
    features['l2_distance'] = l2_distance
    
    # 3. Dot product (absolute value)
    features['dot_product'] = np.dot(source_features, target_features)
    
    # 4. Feature overlap (shared non-zero features)
    source_nonzero = (source_features > 0).astype(int)
    target_nonzero = (target_features > 0).astype(int)
    shared_features = np.sum(source_nonzero * target_nonzero)
    features['shared_features'] = shared_features
    
    # 5. Sum of absolute differences
    features['l1_distance'] = np.sum(np.abs(source_features - target_features))
    
    return features

print("Text feature functions defined successfully.")

In [ ]:
# Compute text features for training set
print("Computing text features for training set...")
text_features_train = []
for idx, row in train_df.iterrows():
    source_features = node_feature_matrix[row['source']]
    target_features = node_feature_matrix[row['target']]
    features = compute_text_features(source_features, target_features)
    text_features_train.append(features)
    if (idx + 1) % 2000 == 0:
        print(f"  Processed {idx + 1} pairs")

text_features_train_df = pd.DataFrame(text_features_train)
print(f"\nText features shape: {text_features_train_df.shape}")
print(f"\nFeature statistics:")
print(text_features_train_df.describe())

In [ ]:
# Compute text features for test set
print("Computing text features for test set...")
text_features_test = []
for idx, row in test_df.iterrows():
    source_features = node_feature_matrix[row['source']]
    target_features = node_feature_matrix[row['target']]
    features = compute_text_features(source_features, target_features)
    text_features_test.append(features)
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1} pairs")

text_features_test_df = pd.DataFrame(text_features_test)
print(f"\nText features test shape: {text_features_test_df.shape}")

### 4.3 Combine All Features

In [ ]:
# Combine all features for training set
X_train = pd.concat([
    graph_features_train_df,
    text_features_train_df
], axis=1)
y_train = train_df['label'].values

print(f"Training features shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"\nFeature columns: {list(X_train.columns)}")
print(f"\nFeature matrix info:")
print(X_train.info())

In [ ]:
# Combine all features for test set
X_test = pd.concat([
    graph_features_test_df,
    text_features_test_df
], axis=1)

print(f"Test features shape: {X_test.shape}")

In [ ]:
# Handle any potential missing values and scale features
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Replace inf values
X_train = X_train.replace([np.inf, -np.inf], 999)
X_test = X_test.replace([np.inf, -np.inf], 999)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled test features shape: {X_test_scaled.shape}")

## 5. Model Training and Comparison
### 5.1 Train-Validation Split

In [ ]:
# Split training data for model validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

print(f"Training split size: {X_train_split.shape}")
print(f"Validation split size: {X_val_split.shape}")
print(f"\nTraining split label distribution:")
print(f"  Negative: {np.sum(y_train_split == 0)}")
print(f"  Positive: {np.sum(y_train_split == 1)}")
print(f"\nValidation split label distribution:")
print(f"  Negative: {np.sum(y_val_split == 0)}")
print(f"  Positive: {np.sum(y_val_split == 1)}")

### 5.2 Model 1: Random Forest

In [ ]:
# Random Forest with hyperparameter tuning
print("Training Random Forest Classifier...")

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_split, y_train_split)

print(f"\nBest Random Forest parameters: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC score: {rf_grid.best_score_:.4f}")

# Validate on validation set
rf_pred_val = rf_grid.predict_proba(X_val_split)[:, 1]
rf_auc = roc_auc_score(y_val_split, rf_pred_val)
print(f"Validation ROC-AUC: {rf_auc:.4f}")

### 5.3 Model 2: Gradient Boosting

In [ ]:
# Gradient Boosting with hyperparameter tuning
print("Training Gradient Boosting Classifier...")

gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=SEED),
    gb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(X_train_split, y_train_split)

print(f"\nBest Gradient Boosting parameters: {gb_grid.best_params_}")
print(f"Best CV ROC-AUC score: {gb_grid.best_score_:.4f}")

# Validate on validation set
gb_pred_val = gb_grid.predict_proba(X_val_split)[:, 1]
gb_auc = roc_auc_score(y_val_split, gb_pred_val)
print(f"Validation ROC-AUC: {gb_auc:.4f}")

### 5.4 Model 3: Logistic Regression

In [ ]:
# Logistic Regression with hyperparameter tuning
print("Training Logistic Regression...")

lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=SEED),
    lr_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train_split, y_train_split)

print(f"\nBest Logistic Regression parameters: {lr_grid.best_params_}")
print(f"Best CV ROC-AUC score: {lr_grid.best_score_:.4f}")

# Validate on validation set
lr_pred_val = lr_grid.predict_proba(X_val_split)[:, 1]
lr_auc = roc_auc_score(y_val_split, lr_pred_val)
print(f"Validation ROC-AUC: {lr_auc:.4f}")

### 5.5 Model Comparison

In [ ]:
# Compare models
models_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'Logistic Regression'],
    'CV ROC-AUC': [rf_grid.best_score_, gb_grid.best_score_, lr_grid.best_score_],
    'Validation ROC-AUC': [rf_auc, gb_auc, lr_auc]
})

print("\nModel Comparison:")
print(models_comparison.to_string())

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models_comparison))
width = 0.35
ax.bar(x - width/2, models_comparison['CV ROC-AUC'], width, label='CV Score')
ax.bar(x + width/2, models_comparison['Validation ROC-AUC'], width, label='Validation Score')
ax.set_ylabel('ROC-AUC Score')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models_comparison['Model'])
ax.legend()
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Get feature importances from Random Forest (best model)
best_rf_model = rf_grid.best_estimator_
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance.head(15).to_string())

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(15), feature_importance.head(15)['importance'])
ax.set_yticks(range(15))
ax.set_yticklabels(feature_importance.head(15)['feature'])
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances (Random Forest)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Make Final Predictions

In [ ]:
# Use the best performing model for predictions
# (Choose based on validation performance)
print("Generating predictions on test set...")

# Use Gradient Boosting if it has the best validation AUC
best_model = gb_grid.best_estimator_
test_predictions = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"Predictions shape: {test_predictions.shape}")
print(f"\nPrediction statistics:")
print(f"Min: {test_predictions.min():.4f}")
print(f"Max: {test_predictions.max():.4f}")
print(f"Mean: {test_predictions.mean():.4f}")
print(f"Std: {test_predictions.std():.4f}")

## 8. Create Submission File

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'ID': range(len(test_predictions)),
    'Predictions': test_predictions
})

print(f"Submission shape: {submission.shape}")
print(f"\nFirst few rows:")
print(submission.head())
print(f"\nLast few rows:")
print(submission.tail())

In [ ]:
# Save submission
submission.to_csv('predictions.csv', index=False)
print("Submission saved to 'predictions.csv'")

## 9. Additional Analysis

In [ ]:
# Analyze prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of predictions
axes[0].hist(test_predictions, bins=50, edgecolor='black')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Test Predictions')

# ROC curve on validation set
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_val_split, gb_pred_val)
axes[1].plot(fpr, tpr, lw=2, label=f'Gradient Boosting (AUC={gb_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Validation Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()